# 漫剧投放账号效能诊断与行动分层

## TL;DR

- 总消耗 207057.29，24h 混合变现 213266.26，加权混合 ROI 为 1.030x。
- 达到消耗 50 门槛的账号 283 个，覆盖 99.4% 消耗；Top 55 账号贡献 64.7% 消耗。
- 五类行动池：核心扩量候选 62、高消耗重点优化 9、小步扩量观察 137、低效清理候选 75、数据不足/低量池 268。
- CTR、CPC、播放率、重算播放成本与混合 ROI 的秩相关均较弱，只能作为待验证线索。

## Context & Methods

使用安全除法、品牌加权聚合、账号消耗 Pareto 和“消耗 × 混合 ROI”规则分层。消耗门槛为 50；有效账号的高消耗边界为 P75；ROI=1 仅表示本口径下变现金额覆盖投放消耗。

## Data

使用同一深度脱敏单月账号表；客户字段不会进入公开网页数据。

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from manga_ad_analysis.io import load_source
from manga_ad_analysis.metrics import brand_summary
from manga_ad_analysis.segmentation import build_pareto, segment_accounts

frame = load_source(PROJECT_ROOT / "data/processed/manga_ad_account_2026_07_anonymized.xlsx")
brands = brand_summary(frame)
segmented, parameters = segment_accounts(frame)
pareto = build_pareto(frame)
print(parameters)

{'spend_floor': 50.0, 'roi_line': 1.0, 'high_spend_quantile': 0.75, 'high_spend_threshold': 735.80492, 'eligible_accounts': 283, 'eligible_spend_coverage': 0.9943377700143088}


## Results

### 1. 品牌规模与效率

In [2]:
columns = ["brand_name", "account_count", "spend", "spend_share", "ctr", "play_rate", "mixed_roi"]
print(brands[columns].round(4).to_string(index=False))

brand_name  account_count      spend  spend_share    ctr  play_rate  mixed_roi
     漫剧品牌C            150 99787.6063       0.4819 0.0256     0.1704     1.0417
     漫剧品牌D            118 39794.2027       0.1922 0.0425     0.1347     1.0363
     漫剧品牌G             73 38396.6365       0.1854 0.0441     0.0828     1.0263
     漫剧品牌F             38 12218.7104       0.0590 0.0280     0.0246     1.0200
     漫剧品牌B             93  7105.6218       0.0343 0.0431     0.1536     0.9408
     漫剧品牌E             29  5152.9035       0.0249 0.1392     0.1847     0.9061
     漫剧品牌H             13  4586.2525       0.0221 0.0158     0.2168     1.0559
     漫剧品牌A             37    15.3535       0.0001 0.0112     0.0041     0.7933


![品牌加权混合 ROI](../images/brand_weighted_roi.png)

### 2. 预算集中度

In [3]:
top_count = int(len(pareto) * 0.10)
top_share = pareto.iloc[top_count - 1]["cumulative_spend_share"]
print(f"Top {top_count} / {len(pareto)} accounts spend share = {top_share:.4%}")
print(pareto.head(10).round(4).to_string(index=False))

Top 55 / 551 accounts spend share = 64.7374%
account_id  account_name brand_name      spend  spend_rank  account_share  spend_share  cumulative_spend_share
   MJ-0482 漫剧品牌G-账号-0482      漫剧品牌G 11382.8634           1         0.0018       0.0550                  0.0550
   MJ-0250 漫剧品牌C-账号-0250      漫剧品牌C  9338.7640           2         0.0036       0.0451                  0.1001
   MJ-0128 漫剧品牌C-账号-0128      漫剧品牌C  5128.3128           3         0.0054       0.0248                  0.1248
   MJ-0310 漫剧品牌F-账号-0310      漫剧品牌F  4083.2158           4         0.0073       0.0197                  0.1446
   MJ-0113 漫剧品牌C-账号-0113      漫剧品牌C  4053.7084           5         0.0091       0.0196                  0.1641
   MJ-0321 漫剧品牌F-账号-0321      漫剧品牌F  4046.2922           6         0.0109       0.0195                  0.1837
   MJ-0204 漫剧品牌C-账号-0204      漫剧品牌C  3749.7315           7         0.0127       0.0181                  0.2018
   MJ-0304 漫剧品牌B-账号-0304      漫剧品牌B  3520.8982           8         

![账号消耗 Pareto](../images/account_pareto.png)

### 3. 五类行动池

In [4]:
segment_counts = segmented["action_segment"].value_counts()
print(segment_counts.to_string())
print()
print("High-spend optimization review queue:")
columns = ["account_id", "account_name", "brand_name", "platform_spend", "mixed_roi_recalc", "play_cost_recalc"]
print(segmented[segmented["action_segment"] == "高消耗重点优化"][columns].sort_values("platform_spend", ascending=False).round(4).to_string(index=False))

action_segment
数据不足/低量池    268
小步扩量观察      137
低效清理候选       75
核心扩量候选       62
高消耗重点优化       9

High-spend optimization review queue:
account_id  account_name brand_name  platform_spend  mixed_roi_recalc  play_cost_recalc
   MJ-0304 漫剧品牌B-账号-0304      漫剧品牌B       3520.8982            0.9421            0.0392
   MJ-0119 漫剧品牌C-账号-0119      漫剧品牌C       2289.5367            0.9986            0.0439
   MJ-0036 漫剧品牌C-账号-0036      漫剧品牌C       1930.8693            0.9606            0.0383
   MJ-0242 漫剧品牌C-账号-0242      漫剧品牌C       1339.8899            0.9221            0.0554
   MJ-0210 漫剧品牌B-账号-0210      漫剧品牌B       1334.5700            0.9687            0.0678
   MJ-0504 漫剧品牌G-账号-0504      漫剧品牌G       1128.9778            0.9060            0.9707
   MJ-0105 漫剧品牌C-账号-0105      漫剧品牌C       1070.3408            0.9886            0.0748
   MJ-0239 漫剧品牌B-账号-0239      漫剧品牌B        806.6755            0.9644            0.0728
   MJ-0112 漫剧品牌C-账号-0112      漫剧品牌C        770.1654            0.9816     

![账号行动分层](../images/action_segments.png)

### 4. 相关性探索（非因果）

In [5]:
eligible = segmented[segmented["is_eligible"]]
fields = ["ctr_recalc", "cpc_recalc", "play_rate_recalc", "play_cost_recalc", "mixed_roi_recalc"]
correlations = eligible[fields].corr(method="spearman")
print(correlations[["mixed_roi_recalc"]].drop(index="mixed_roi_recalc").round(4).to_string())

                  mixed_roi_recalc
ctr_recalc                  0.0272
cpc_recalc                 -0.0306
play_rate_recalc            0.0296
play_cost_recalc           -0.1279


![混合 ROI 相关性](../images/roi_correlations.png)

## Takeaways

- 预算高度集中，优先把人工核查放在高消耗账号而不是平均覆盖全部 551 个账号。
- 9 个“高消耗重点优化”账号应先检查素材、计划配置和落地链路；62 个“核心扩量候选”只建议小幅、带上限地验证。
- 268 个低量账号不适合直接判定好坏，应先补足样本。
- 单月横截面和弱相关性不足以支持“某指标导致 ROI 变化”的结论；下一步需补多月、素材/计划粒度和利润口径。